In [1]:
import torch
import torch.nn as nn
import timm  # Library for state-of-the-art image models
import pandas as pd
from pathlib import Path

c:\Users\misog\Documents\FUB_sems\summer26\Computer-VIsion-2026\cv_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device Name:", torch.cuda.get_device_name(0))

CUDA Available: False


In [3]:
BASE_PATH = Path.cwd()
TRAIN_CSV_PATH: Path = BASE_PATH / "train.csv"
TEST_CSV_PATH: Path = BASE_PATH / "test.csv"
IMG_SIZE_CSV_PATH: Path = BASE_PATH / "img_size.csv"
TRAIN_IMG_PATH: Path = BASE_PATH / "train" / "train_raw" 
TEST_IMG_PATH: Path = BASE_PATH / "test" / "test_raw"

In [4]:
class ChestXrayClassifier(nn.Module):
    def __init__(self, num_classes=14, pretrained=True):
        super(ChestXrayClassifier, self).__init__()
        
        # Load a pretrained EfficientNet backbone
        # 'pretrained=True' loads weights trained on ImageNet (millions of natural images)
        self.backbone = timm.create_model('efficientnet_b4', pretrained=pretrained)
        
        # EfficientNet-B4 outputs 1792 features before its final classification layer
        in_features = self.backbone.num_features
        
        # Replace the final classification head
        # We map those 1792 features directly to our 14 abnormality classes
        self.backbone.classifier = nn.Linear(in_features, num_classes)
        
    def forward(self, x):
        # Returns raw outputs (logits) for each of the 14 classes
        return self.backbone(x)

# Quick sanity check
model = ChestXrayClassifier(num_classes=14)
print(f"Model successfully initialized!")

Model successfully initialized!


In [5]:
# Force it to CPU since you have a powerful Intel i5 processor
DEVICE = torch.device('cpu')
print(f"Using device: {DEVICE}")

# Initialize model and put it on the CPU
model = ChestXrayClassifier(num_classes=14)
model.to(DEVICE)

Using device: cpu


ChestXrayClassifier(
  (backbone): EfficientNet(
    (conv_stem): Conv2d(3, 48, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (bn1): BatchNormAct2d(
      48, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True
      (drop): Identity()
      (act): SiLU(inplace=True)
    )
    (blocks): Sequential(
      (0): Sequential(
        (0): DepthwiseSeparableConv(
          (conv_dw): Conv2d(48, 48, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=48, bias=False)
          (bn1): BatchNormAct2d(
            48, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True
            (drop): Identity()
            (act): SiLU(inplace=True)
          )
          (aa): Identity()
          (se): SqueezeExcite(
            (conv_reduce): Conv2d(48, 12, kernel_size=(1, 1), stride=(1, 1))
            (act1): SiLU(inplace=True)
            (conv_expand): Conv2d(12, 48, kernel_size=(1, 1), stride=(1, 1))
            (gate): Sigmoid()
  

In [6]:
from torch.utils.data import Dataset, DataLoader
import cv2 as cv
import numpy as np

class ChestXrayDataset(Dataset):
    def __init__(self, df, img_dir):
        self.df = df
        self.img_dir = img_dir
        self.image_ids = df['image_id'].values
        
        # Select just the class_0 to class_13 columns as our targets
        # (We exclude 'image_id' and 'no_finding' because no_finding is calculated implicitly)
        self.target_cols = [f'class_{i}' for i in range(14)]
        self.targets = df[self.target_cols].values

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        # 1. Load the image
        img_id = self.image_ids[idx]
        img_path = self.img_dir / f"{img_id}.png"
        image = cv.imread(str(img_path))
        
        # 2. Convert BGR to RGB and normalize pixel values to [0, 1]
        image = cv.cvtColor(image, cv.COLOR_BGR2RGB)
        image = image.astype(np.float32) / 255.0
        
        # 3. Change shape from HWC (Height, Width, Channels) to PyTorch's CHW
        image = np.transpose(image, (2, 0, 1))
        
        # 4. Convert both to PyTorch Tensors
        image_tensor = torch.tensor(image, dtype=torch.float32)
        target_tensor = torch.tensor(self.targets[idx], dtype=torch.float32)
        
        return image_tensor, target_tensor

In [7]:
classification_df = pd.read_csv(BASE_PATH / "train" / "train_multilabel_targets.csv")

In [8]:
import torch.optim as optim
from pathlib import Path

# --- Configuration ---
BATCH_SIZE = 4
EPOCHS = 5
LEARNING_RATE = 1e-4
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

# --- Initialize Data Components ---
# (Replace classification_df with your actual processed target matrix DataFrame)
train_dataset = ChestXrayDataset(classification_df, BASE_PATH / "train" / "train_512")
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

# --- Initialize Model Core ---
model = ChestXrayClassifier(num_classes=14)
model.to(DEVICE)

# --- Define Optimization Functions ---
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)

Using device: cpu


In [ ]:
from tqdm.auto import tqdm

for epoch in range(EPOCHS):
    model.train()  # Put model in training mode
    running_loss = 0.0
    
    # Wrap train_loader in tqdm for a clean visual progress bar
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    
    for images, targets in progress_bar:
        # Move tensors to GPU/CPU memory
        images = images.to(DEVICE)
        targets = targets.to(DEVICE)
        
        # 1. Clear previous gradients
        optimizer.zero_grad()
        
        # 2. Forward pass: Calculate predictions (logits)
        outputs = model(images)
        
        # 3. Calculate Loss
        loss = criterion(outputs, targets)
        
        # 4. Backward pass: Calculate gradients
        loss.backward()
        
        # 5. Step: Adjust weights based on gradients
        optimizer.step()
        
        # Track historical loss statistics
        running_loss += loss.item()
        progress_bar.set_postfix({'loss': f'{loss.item():.4f}'})
        
    epoch_loss = running_loss / len(train_loader)
    print(f"==> Epoch {epoch+1} Complete. Average Loss: {epoch_loss:.4f}\n")

# Save your trained weights so you don't lose progress!
torch.save(model.state_dict(), "chest_xray_baseline_weights.pth")
print("Model training complete and weights successfully saved!")

Epoch 1/5:   0%|          | 0/2143 [00:00<?, ?it/s]

Epoch 1/5:  37%|███▋      | 799/2143 [1:17:30<2:40:36,  7.17s/it, loss=0.1787]